<a href="https://colab.research.google.com/github/yosie111/cadcam/blob/main/stage3_cadcam_bridge_heb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# שלב 3 — ארכיטקטורה סופית: Bridge + Facade + Adapter

**מקור:** *Design Patterns Explained*, פרק 12 — "Solving the CAD/CAM Problem with Patterns"

---

## תהליך הגזירה — "Thinking in Patterns"

### שלב 1: זיהוי הדפוסים המועמדים
הספר מזהה ארבעה דפוסים בתחום הבעיה: **Abstract Factory**, **Adapter**, **Bridge**, **Facade**.

### שלב 2a: מי יוצר הקשר למי?
- **Abstract Factory** — נפסל כדפוס ראשי. הוא תלוי באובייקטים שהדפוסים האחרים יגדירו.
- **Bridge vs Adapter** — Bridge מגדיר את הממשק שאליו Adapter צריך להתאים. Bridge יוצר הקשר ל-Adapter.
- **Bridge vs Facade** — Bridge מגדיר את הממשק שה-Facade צריך לספק. Bridge יוצר הקשר ל-Facade.

> *"The Bridge is the winner"* — הוא הדפוס הבכיר (**seniormost**) שיוצר הקשר לכל השאר.

### שלב 2b: יישום Bridge
- **Abstraction** = Feature (מה שמערכת המומחה רואה)
- **Implementor** = FeatureImp (איך מחלצים נתונים מ-CAD/CAM)
- Feature **מחזיק** FeatureImp בהרכבה — לא יודע אם זה V1 או V2

### שלב 2c: יישום Facade (ל-V1)
V1 הוא פרוצדורלי ומסורבל. V1Facade עוטף אותו בממשק OO נקי. V1Imp משתמש ב-V1Facade.

### שלב 2d: יישום Adapter (ל-V2)
V2 מחזיר OOGFeature עם שמות שונים. V2Imp **מתאים (adapt)** את OOGFeature לממשק FeatureImp.

### שלב 2e: Abstract Factory — לא נדרש
> *"Model can easily encapsulate the rules of creation."* — Model יודע אם זה V1 או V2.

---

## תרשים מבנה (Figure 12-11)

```
    ╔═══════════════════════════════════════════════════════════════════╗
    ║  מערכת המומחה (לא יודעת כלום על V1 או V2)                       ║
    ╚═════════════════════════════╤═════════════════════════════════════╝
                                  │ list[Feature]
    ┌─────────────────────────────┴──────────────────────────────┐
    │                    Feature (ABC)                            │
    │                 get_x(), get_y()                            │
    │                 _imp: FeatureImp    ← Bridge: composition   │
    │                       △                                     │
    │     ┌─────────┬───────┼──────────┬──────────┐              │
    │  SlotFeature Hole  Cutout    Special   Irregular            │
    └────────────────────────────────────────────────────────────┘
                                  │
                    ╔═════════════╧═════════════╗
                    ║   FeatureImp (ABC)         ║
                    ║   get_x(), get_y(),        ║
                    ║   get_width(), ...          ║
                    ╚═══════╤═══════════╤═══════╝
                            │           │
                  ┌─────────┴──┐  ┌─────┴──────┐
                  │   V1Imp    │  │   V2Imp     │
                  │  (Facade)  │  │  (Adapter)  │
                  └─────┬──────┘  └──────┬──────┘
                        │                │
                  ┌─────┴──────┐  ┌──────┴──────┐
                  │ V1Facade   │  │ OOGFeature  │
                  │ (wraps V1) │  │ (V2 objects)│
                  └─────┬──────┘  └─────────────┘
                        │
                  ┌─────┴──────┐
                  │ V1 System  │
                  │ (procedural│
                  │  functions)│
                  └────────────┘
```

### השוואת ספירת מחלקות

| | פרק 4 (שלב 2) | פרק 12 (שלב 3) |
|:--|:--:|:--:|
| V1 + V2 | **16** מחלקות | **10** מחלקות |
| +V3 | **21** (+5) | **11** (+1) |
| +V4 | **26** (+5) | **12** (+1) |
| נוסחה | 6 + 5N | 7 + N |

---

## אחריות כל מחלקה

| מחלקה | דפוס | תפקיד |
|:------|:-----|:------|
| `Feature` | Bridge (Abstraction) | ממשק שמערכת המומחה רואה. מחזיק `_imp` בהרכבה. |
| `SlotFeature` | RefinedAbstraction | מוסיף `get_width()`, `get_height()` — מאציל ל-`_imp`. |
| `FeatureImp` | Bridge (Implementor) | ממשק אחיד לחילוץ נתונים מכל גרסת CAD/CAM. |
| `V1Imp` | ConcreteImplementor | מממש `FeatureImp` דרך `V1Facade`. |
| `V2Imp` | ConcreteImplementor + **Adapter** | מממש `FeatureImp` ע"י התאמת `OOGFeature`. |
| `V1Facade` | **Facade** | עוטף את V1 הפרוצדורלי בממשק OO נקי. |
| `Model` (loader) | — | מרכיב Feature + Imp. לא צריך Abstract Factory. |

---
## שכבה 1: מערכות ה-CAD/CAM (נתון — לא ניתן לשנות)

In [ ]:
# ════════════════════════════════════════════════════════════
# שכבה 1: מערכות ה-CAD/CAM (נתון — לא ניתן לשנות)
# ════════════════════════════════════════════════════════════

from __future__ import annotations
from abc import ABC, abstractmethod

# ── V1: ספרייה פרוצדורלית (מקוצר — אותו מסד נתונים כמו שלב 1 ו-2) ──

FID=0; FTYPE=1; X=2; Y=3; WIDTH=4; HEIGHT=5; ANGLE=6; DEPTH=7; RADIUS=8; SPECIAL_CODE=9

_V1_DATABASE: dict[str, dict] = {
    "PART-001": {
        "sheet_width": 200.0, "sheet_height": 150.0, "material": "Aluminum-6061",
        "features": [
            ( 1, "slot",     10.0,  30.0,  40.0,  8.0,  0.0, 2.0, None,  None),
            ( 2, "slot",     10.0,  60.0,  40.0,  8.0,  0.0, 2.0, None,  None),
            ( 3, "hole",     30.0,  60.0,  12.0, 12.0,  0.0, 2.0,  6.0,  None),
            ( 4, "hole",    120.0,  75.0,  30.0, 30.0,  0.0, 2.0, 15.0,  None),
            ( 5, "cutout",   80.0,  40.0,  35.0, 25.0,  0.0, 2.0, None,  None),
            ( 6, "cutout",  130.0,  90.0,  25.0, 25.0, 45.0, 2.0, None,  None),
            ( 7, "special",  90.0,  30.0,  20.0, 20.0,  0.0, 2.0, None,  "ELEC-OUTLET"),
            ( 8, "irregular",140.0, 110.0,  25.0, 15.0, 30.0, 2.0, None,  None),
        ],
    },
}

_open_handles: dict[int, str] = {}
_next_handle: int = 0

def v1_open_model(name: str) -> int:
    global _next_handle
    if name not in _V1_DATABASE: raise ValueError(f"V1: {name!r} not found")
    _next_handle += 1; _open_handles[_next_handle] = name; return _next_handle
def v1_close_model(h: int) -> None: _open_handles.pop(h, None)
def _v1m(h): return _V1_DATABASE[_open_handles[h]]
def _v1f(h, fid):
    for r in _v1m(h)["features"]:
        if r[FID] == fid: return r
    raise ValueError(f"V1: feature {fid} not found")
def v1_get_num_features(h): return len(_v1m(h)["features"])
def v1_get_feature_id(h, i): return _v1m(h)["features"][i][FID]
def v1_get_feature_type(h, i): return _v1m(h)["features"][i][FTYPE]
def v1_get_x(h, fid, ftype): return _v1f(h, fid)[X]
def v1_get_y(h, fid, ftype): return _v1f(h, fid)[Y]
def v1_get_w(h, fid, ftype): return _v1f(h, fid)[WIDTH]
def v1_get_h(h, fid, ftype): return _v1f(h, fid)[HEIGHT]
def v1_get_angle(h, fid, ftype): return _v1f(h, fid)[ANGLE]
def v1_get_depth(h, fid, ftype): return _v1f(h, fid)[DEPTH]
def v1_get_radius(h, fid): return _v1f(h, fid)[RADIUS]
def v1_get_special_code(h, fid): return _v1f(h, fid)[SPECIAL_CODE]


# ── V2: OOG (Object Oriented Geometry) ──

class OOGFeature:
    def __init__(self, ftype: str, x: float, y: float,
                 length: float, width: float, angle: float = 0.0,
                 depth: float = 1.0, radius: float | None = None,
                 special_code: str | None = None, edge_type: str = "square"):
        self._type = ftype; self._x = x; self._y = y
        self._length = length; self._width = width
        self._angle = angle; self._depth = depth
        self._radius = radius; self._special_code = special_code
        self._edge_type = edge_type

    def my_type(self) -> str: return self._type
    def get_x(self) -> float: return self._x
    def get_y(self) -> float: return self._y
    def get_length(self) -> float: return self._length
    def get_width(self) -> float: return self._width
    def get_angle(self) -> float: return self._angle
    def get_depth(self) -> float: return self._depth
    def get_radius(self) -> float: return self._radius or self._length / 2
    def get_special_code(self) -> str: return self._special_code or "?"
    def get_edge_type(self) -> str: return self._edge_type

class OOGModel:
    def __init__(self, name, sw, sh, mat, features):
        self._name=name; self._sw=sw; self._sh=sh; self._mat=mat; self._f=features
    def get_model_name(self): return self._name
    def get_elements(self) -> list[OOGFeature]: return list(self._f)

_V2_DATABASE = {
    "PART-001": OOGModel("PART-001", 200, 150, "Aluminum-6061", [
        OOGFeature("slot",     10.0,  30.0, 40.0,  8.0,  0.0, 2.0, edge_type="rounded"),
        OOGFeature("slot",     10.0,  60.0, 40.0,  8.0,  0.0, 2.0, edge_type="rounded"),
        OOGFeature("hole",     30.0,  60.0, 12.0, 12.0,  0.0, 2.0, radius=6.0),
        OOGFeature("hole",    120.0,  75.0, 30.0, 30.0,  0.0, 2.0, radius=15.0),
        OOGFeature("cutout",   80.0,  40.0, 35.0, 25.0,  0.0, 2.0, edge_type="squared"),
        OOGFeature("cutout",  130.0,  90.0, 25.0, 25.0, 45.0, 2.0, edge_type="rounded"),
        OOGFeature("special",  90.0,  30.0, 20.0, 20.0,  0.0, 2.0, special_code="ELEC-OUTLET"),
        OOGFeature("irregular",140.0, 110.0, 25.0, 15.0, 30.0, 2.0),
    ]),
}

def v2_open_model(name: str) -> OOGModel:
    if name not in _V2_DATABASE: raise ValueError(f"V2: {name!r} not found")
    return _V2_DATABASE[name]

print("שכבה 1 מוכנה: V1 (פרוצדורלי) + V2 (OOG)")

שכבה 1 מוכנה: V1 (פרוצדורלי) + V2 (OOG)


---
## שכבה 2: FeatureImp — צד המימוש של ה-Bridge (Implementor)
**למה הוא קיים:** FeatureImp מגדיר ממשק **אחיד** לחילוץ נתונים מכל גרסת CAD/CAM.
הוא מכיל את כל השדות שאי-פעם feature כלשהו עשוי לבקש.
Feature מאציל אליו בלי לדעת אם מאחוריו V1, V2, או V3.

In [ ]:
# ════════════════════════════════════════════════════════════
# שכבה 2: FeatureImp — הממשק של צד המימוש (Implementor)
# ════════════════════════════════════════════════════════════

class FeatureImp(ABC):
    """Implementor בדפוס Bridge — ממשק אחיד לחילוץ נתונים."""

    @abstractmethod
    def get_x(self) -> float: ...

    @abstractmethod
    def get_y(self) -> float: ...

    @abstractmethod
    def get_width(self) -> float: ...

    @abstractmethod
    def get_height(self) -> float: ...

    @abstractmethod
    def get_angle(self) -> float: ...

    @abstractmethod
    def get_depth(self) -> float: ...

    @abstractmethod
    def get_radius(self) -> float: ...

    @abstractmethod
    def get_special_code(self) -> str: ...


print("שכבה 2 מוכנה: FeatureImp (ממשק מימוש)")

שכבה 2 מוכנה: FeatureImp (ממשק מימוש)


---
## שכבה 3a: V1Facade + V1Imp — Facade עוטף את V1 הפרוצדורלי
**V1Facade** — מפשט את הפונקציות הפרוצדורליות המסורבלות לממשק OO אחיד. שומר handle + feature_id.
**V1Imp** — מממש את `FeatureImp` על ידי האצלה ל-`V1Facade`.

In [ ]:
# ════════════════════════════════════════════════════════════
# שכבה 3a: V1Facade + V1Imp
# ════════════════════════════════════════════════════════════

class V1Facade:
    """Facade: עוטף את ספריית V1 הפרוצדורלית בממשק OO.
    שומר handle + feature_id — מידע שרק V1 צריך."""

    def __init__(self, handle: int, feature_id: int, feature_type: str):
        self._h = handle
        self._fid = feature_id
        self._ftype = feature_type

    def get_x(self) -> float:           return v1_get_x(self._h, self._fid, self._ftype)
    def get_y(self) -> float:           return v1_get_y(self._h, self._fid, self._ftype)
    def get_width(self) -> float:       return v1_get_w(self._h, self._fid, self._ftype)
    def get_height(self) -> float:      return v1_get_h(self._h, self._fid, self._ftype)
    def get_angle(self) -> float:       return v1_get_angle(self._h, self._fid, self._ftype)
    def get_depth(self) -> float:       return v1_get_depth(self._h, self._fid, self._ftype)
    def get_radius(self) -> float:      return v1_get_radius(self._h, self._fid)
    def get_special_code(self) -> str:  return v1_get_special_code(self._h, self._fid)


class V1Imp(FeatureImp):
    """ConcreteImplementor: מממש FeatureImp דרך V1Facade."""

    def __init__(self, facade: V1Facade):
        self._facade = facade

    def get_x(self) -> float:           return self._facade.get_x()
    def get_y(self) -> float:           return self._facade.get_y()
    def get_width(self) -> float:       return self._facade.get_width()
    def get_height(self) -> float:      return self._facade.get_height()
    def get_angle(self) -> float:       return self._facade.get_angle()
    def get_depth(self) -> float:       return self._facade.get_depth()
    def get_radius(self) -> float:      return self._facade.get_radius()
    def get_special_code(self) -> str:  return self._facade.get_special_code()


print("שכבה 3a מוכנה: V1Facade + V1Imp")

שכבה 3a מוכנה: V1Facade + V1Imp


---
## שכבה 3b: V2Imp — Adapter מתאים את OOGFeature לממשק שלנו
**V2Imp** שומר הפניה ל-`OOGFeature` ומאציל אליו **עם מיפוי שמות**:
- `get_length()` → `get_width()` שלנו
- `get_width()` → `get_height()` שלנו

מיפוי השמות נמצא **במקום אחד בלבד** — ב-V2Imp. לא מפוזר על 5 מחלקות כמו בפרק 4.

In [ ]:
# ════════════════════════════════════════════════════════════
# שכבה 3b: V2Imp — Adapter
# ════════════════════════════════════════════════════════════

class V2Imp(FeatureImp):
    """ConcreteImplementor + Adapter: מתאים OOGFeature ל-FeatureImp."""

    def __init__(self, oog: OOGFeature):
        self._oog = oog

    def get_x(self) -> float:           return self._oog.get_x()
    def get_y(self) -> float:           return self._oog.get_y()
    def get_width(self) -> float:       return self._oog.get_length()    # ← מיפוי שמות!
    def get_height(self) -> float:      return self._oog.get_width()     # ← מיפוי שמות!
    def get_angle(self) -> float:       return self._oog.get_angle()
    def get_depth(self) -> float:       return self._oog.get_depth()
    def get_radius(self) -> float:      return self._oog.get_radius()
    def get_special_code(self) -> str:  return self._oog.get_special_code()


print("שכבה 3b מוכנה: V2Imp (Adapter)")

שכבה 3b מוכנה: V2Imp (Adapter)


---
## שכבה 4: Feature — צד ההפשטה של ה-Bridge (Abstraction)
**Feature** הוא ה-Abstraction. הוא מחזיק `FeatureImp` **בהרכבה** (composition) ומאציל אליו.
Feature **לא יודע כלום** על V1 או V2 — רק על ה-imp שלו.

`SlotFeature`, `HoleFeature` וכו' הם **RefinedAbstractions** — מוסיפים מתודות ייחודיות לסוג,
אבל עדיין מאצילים ל-`_imp`. **אין בהם שום מידע על הגרסה.**

In [ ]:
# ════════════════════════════════════════════════════════════
# שכבה 4: Feature — צד ההפשטה (Abstraction)
# ════════════════════════════════════════════════════════════

class Feature(ABC):
    """Abstraction: מחזיק FeatureImp בהרכבה — לא יודע אם זה V1 או V2."""

    def __init__(self, imp: FeatureImp):
        self._imp = imp    # ← זה ה-Bridge: הפשטה מחזיקה מימוש

    @abstractmethod
    def feature_type(self) -> str: ...

    def get_x(self) -> float: return self._imp.get_x()
    def get_y(self) -> float: return self._imp.get_y()


class SlotFeature(Feature):
    """RefinedAbstraction: slot."""
    def feature_type(self) -> str: return "slot"
    def get_width(self) -> float:  return self._imp.get_width()
    def get_height(self) -> float: return self._imp.get_height()
    def get_angle(self) -> float:  return self._imp.get_angle()
    def get_depth(self) -> float:  return self._imp.get_depth()


class HoleFeature(Feature):
    """RefinedAbstraction: hole."""
    def feature_type(self) -> str: return "hole"
    def get_radius(self) -> float: return self._imp.get_radius()
    def get_depth(self) -> float:  return self._imp.get_depth()


class CutoutFeature(Feature):
    """RefinedAbstraction: cutout."""
    def feature_type(self) -> str: return "cutout"
    def get_width(self) -> float:  return self._imp.get_width()
    def get_height(self) -> float: return self._imp.get_height()
    def get_angle(self) -> float:  return self._imp.get_angle()


class SpecialFeature(Feature):
    """RefinedAbstraction: special."""
    def feature_type(self) -> str: return "special"
    def get_special_code(self) -> str: return self._imp.get_special_code()


class IrregularFeature(Feature):
    """RefinedAbstraction: irregular."""
    def feature_type(self) -> str: return "irregular"
    def get_bbox_width(self) -> float:  return self._imp.get_width()
    def get_bbox_height(self) -> float: return self._imp.get_height()


print("שכבה 4 מוכנה: Feature + 5 RefinedAbstractions")

שכבה 4 מוכנה: Feature + 5 RefinedAbstractions


---
## שכבה 5: Model — טוען ומרכיב Feature + Imp
Model יודע אם זה V1 או V2 ויוצר את ה-imp הנכון.
**Abstract Factory לא נדרש** — הספר: *"Model can easily encapsulate the rules of creation."*

In [ ]:
# ════════════════════════════════════════════════════════════
# שכבה 5: Model — טוען ומרכיב Feature + Imp
# ════════════════════════════════════════════════════════════

FEATURE_CLASS_MAP: dict[str, type[Feature]] = {
    "slot": SlotFeature, "hole": HoleFeature, "cutout": CutoutFeature,
    "special": SpecialFeature, "irregular": IrregularFeature,
}


def load_v1_model(model_name: str) -> tuple[int, list[Feature]]:
    """V1: יוצר V1Facade → V1Imp → Feature לכל feature."""
    handle = v1_open_model(model_name)
    features: list[Feature] = []

    for i in range(v1_get_num_features(handle)):
        fid = v1_get_feature_id(handle, i)
        ftype = v1_get_feature_type(handle, i)

        facade = V1Facade(handle, fid, ftype)   # Facade עוטף V1
        imp = V1Imp(facade)                       # Imp משתמש ב-Facade
        features.append(FEATURE_CLASS_MAP[ftype](imp))  # Feature מחזיק Imp

    return handle, features


def load_v2_model(model_name: str) -> list[Feature]:
    """V2: יוצר V2Imp (Adapter) → Feature לכל OOGFeature."""
    oog_model = v2_open_model(model_name)
    features: list[Feature] = []

    for oog in oog_model.get_elements():
        ftype = oog.my_type()
        imp = V2Imp(oog)    # Adapter מתאים OOGFeature
        features.append(FEATURE_CLASS_MAP[ftype](imp))

    return features


print("שכבה 5 מוכנה: load_v1_model(), load_v2_model()")

שכבה 5 מוכנה: load_v1_model(), load_v2_model()


---
## שכבה 6: מערכת המומחה — לא יודעת כלום על V1 או V2

In [ ]:
# ════════════════════════════════════════════════════════════
# שכבה 6: מערכת המומחה
# ════════════════════════════════════════════════════════════

def expert_system_process(features: list[Feature]) -> list[str]:
    """מערכת המומחה — עובדת רק עם Feature. לא יודעת מאיפה הנתונים."""
    commands: list[str] = []
    for feat in features:
        x, y = feat.get_x(), feat.get_y()

        if isinstance(feat, SlotFeature):
            commands.append(f"NC: CUT_SLOT at ({x},{y}) w={feat.get_width()} h={feat.get_height()} angle={feat.get_angle()} depth={feat.get_depth()}")
        elif isinstance(feat, HoleFeature):
            commands.append(f"NC: DRILL_HOLE at ({x},{y}) r={feat.get_radius()} depth={feat.get_depth()}")
        elif isinstance(feat, CutoutFeature):
            commands.append(f"NC: PUNCH_CUTOUT at ({x},{y}) w={feat.get_width()} h={feat.get_height()} angle={feat.get_angle()}")
        elif isinstance(feat, SpecialFeature):
            commands.append(f"NC: PUNCH_SPECIAL at ({x},{y}) code={feat.get_special_code()}")
        elif isinstance(feat, IrregularFeature):
            commands.append(f"NC: MULTI_CUT at ({x},{y}) bbox={feat.get_bbox_width()}x{feat.get_bbox_height()}")
    return commands

print("שכבה 6 מוכנה: expert_system_process()")

שכבה 6 מוכנה: expert_system_process()


---
## דמו + בדיקות

In [ ]:
# ════════════════════════════════════════════════════════════
# דמו + בדיקות
# ════════════════════════════════════════════════════════════

print("═" * 60)
print("  V1 ו-V2 דרך אותה ארכיטקטורה (Bridge + Facade + Adapter)")
print("═" * 60)

# ── V1 ──
v1_handle, v1_features = load_v1_model("PART-001")
print(f"\nV1: {len(v1_features)} features")
for f in v1_features:
    print(f"  {f.__class__.__name__:<18} at ({f.get_x()},{f.get_y()})  imp={f._imp.__class__.__name__}")

v1_commands = expert_system_process(v1_features)

# ── V2 ──
v2_features = load_v2_model("PART-001")
print(f"\nV2: {len(v2_features)} features")
for f in v2_features:
    print(f"  {f.__class__.__name__:<18} at ({f.get_x()},{f.get_y()})  imp={f._imp.__class__.__name__}")

v2_commands = expert_system_process(v2_features)

# ── NC ──
print(f"\nפקודות NC (זהות מ-V1 ומ-V2):")
for cmd in v1_commands:
    print(f"  {cmd}")

# ── ספירה ──
print(f"\n{'─'*60}")
print(f"  מחלקות: 6 הפשטות + 3 מימושים + 1 Facade = 10")
print(f"  V3 חדש = +1 מחלקה בלבד (V3Imp)!")

# ── בדיקות ──
print(f"\n{'─'*60}")
print("  בדיקות...")

assert len(v1_features) == len(v2_features) == 8
assert isinstance(v1_features[0]._imp, V1Imp)
assert isinstance(v2_features[0]._imp, V2Imp)
assert isinstance(v1_features[0]._imp._facade, V1Facade)
assert isinstance(v2_features[0]._imp._oog, OOGFeature)
assert v1_features[0].get_x() == v2_features[0].get_x() == 10.0
assert v1_features[0].get_width() == v2_features[0].get_width() == 40.0
assert v1_features[2].get_radius() == v2_features[2].get_radius() == 6.0
assert v1_features[6].get_special_code() == v2_features[6].get_special_code() == "ELEC-OUTLET"
assert v1_commands == v2_commands, "NC commands must be identical!"
v1_close_model(v1_handle)
assert len(_open_handles) == 0

print("\n  כל הבדיקות עברו בהצלחה.")
print(f"{'═'*60}")

════════════════════════════════════════════════════════════
  V1 ו-V2 דרך אותה ארכיטקטורה (Bridge + Facade + Adapter)
════════════════════════════════════════════════════════════

V1: 8 features
  SlotFeature        at (10.0,30.0)  imp=V1Imp
  SlotFeature        at (10.0,60.0)  imp=V1Imp
  HoleFeature        at (30.0,60.0)  imp=V1Imp
  HoleFeature        at (120.0,75.0)  imp=V1Imp
  CutoutFeature      at (80.0,40.0)  imp=V1Imp
  CutoutFeature      at (130.0,90.0)  imp=V1Imp
  SpecialFeature     at (90.0,30.0)  imp=V1Imp
  IrregularFeature   at (140.0,110.0)  imp=V1Imp

V2: 8 features
  SlotFeature        at (10.0,30.0)  imp=V2Imp
  SlotFeature        at (10.0,60.0)  imp=V2Imp
  HoleFeature        at (30.0,60.0)  imp=V2Imp
  HoleFeature        at (120.0,75.0)  imp=V2Imp
  CutoutFeature      at (80.0,40.0)  imp=V2Imp
  CutoutFeature      at (130.0,90.0)  imp=V2Imp
  SpecialFeature     at (90.0,30.0)  imp=V2Imp
  IrregularFeature   at (140.0,110.0)  imp=V2Imp

פקודות NC (זהות מ-V1 ומ-V2)

---

## איזו שונות מבודדת כל דפוס?

| דפוס | שונות שהוא מבודד | מה משתנה | מה נשאר יציב |
|:-----|:-----------------|:---------|:-------------|
| **Bridge** | הפרדה בין *מה* ל*איך* | גרסת CAD/CAM (V1, V2, V3...) | סוגי ה-feature (slot, hole...) |
| **Facade** | פישוט ממשק מורכב | הקריאות הפרוצדורליות של V1 | ממשק FeatureImp האחיד |
| **Adapter** | התאמת ממשק קיים | שמות OOGFeature השונים | ממשק FeatureImp האחיד |
| **~~Abstract Factory~~** | ~~יצירת משפחות אובייקטים~~ | ~~לא נדרש~~ | Model מספיק |

### מבט סופי
> *"Features are either slot features, hole features, cutout features, irregular features, or special features. All features contain an implementation which is either a V1 implementation or a V2 implementation. V1 implementations use a V1 Facade to access the V1 system while V2 implementations adapt an OOGFeature. That's it."*
>
> — פרק 12, עמוד 215